# HW4: Dimensionality Reduction

In this homework you will implement dimensionality reduction techniques from scratch:

1. **PCA via Eigendecomposition** — classical approach using the covariance matrix
2. **PCA via SVD** — numerically stable alternative, and its equivalence to eigendecomposition
3. **Linear Autoencoder** — neural network approach, equivalence to PCA with linear activations
4. **Non-linear Autoencoder** — complex manifold learning with ReLU activations

**Requirements:** `numpy`, `matplotlib`, `torch`

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.optim as optim

from dimred_helpers import (
    # Data generation
    generate_high_dim_blobs,
    generate_swiss_roll,
    # Visualization
    plot_2d_projection,
    plot_explained_variance,
    plot_reconstruction_comparison,
    plot_pca_eigen_vs_svd,
    plot_autoencoder_training,
    plot_latent_space,
    plot_swiss_roll_3d,
    plot_manifold_comparison,
)

from dimred_tests import (
    check_pca_eigen,
    check_pca_svd,
    check_linear_autoencoder,
    check_nonlinear_autoencoder,
    run_single_test,
    run_tests,
)

np.random.seed(42)
torch.manual_seed(42)
plt.rcParams["figure.figsize"] = [10, 6]
plt.rcParams["font.size"] = 12

## Section 2: PCA Building Blocks

**Principal Component Analysis (PCA)** finds the directions of maximum variance in the data.

The key steps are:
1. **Center** the data (subtract the mean)
2. Compute the **covariance matrix** $C = \frac{1}{n-1} X_c^T X_c$
3. Find the **eigenvectors** of $C$ (these are the principal components)
4. **Project** the data onto the top-$k$ eigenvectors

Alternatively, we can use the **Singular Value Decomposition (SVD)** of the centered data matrix directly, which is numerically more stable because it avoids forming $X^T X$ (which squares the condition number).

Let's build each step as a standalone function before combining them into a full PCA class.

### Exercise 2.1: Center Data

Subtract the column-wise mean from the data matrix.

$$X_c = X - \bar{X}$$

where $\bar{X}$ is the mean of each feature (column).

In [ ]:
try:

    def center_data(X):
        """Center the data by subtracting the column-wise mean.

        Parameters
        ----------
        X : np.ndarray of shape (n_samples, n_features)

        Returns
        -------
        X_centered : np.ndarray of shape (n_samples, n_features)
        mean : np.ndarray of shape (n_features,)
        """
        # YOUR CODE HERE
        raise NotImplementedError("Implement center_data")

    # Quick test
    X_test = np.array([[1.0, 2.0], [3.0, 4.0], [5.0, 6.0]])
    X_c, mu = center_data(X_test)
    assert X_c.shape == X_test.shape, f"Expected shape {X_test.shape}, got {X_c.shape}"
    assert np.allclose(X_c.mean(axis=0), 0), f"Centered data mean should be ~0, got {X_c.mean(axis=0)}"
    assert np.allclose(mu, [3.0, 4.0]), f"Mean should be [3, 4], got {mu}"
    print("center_data tests passed! ✓")
except Exception as e:
    print(f"⚠️ Skipped (not yet implemented or error): {e}")

### Exercise 2.2: Covariance Matrix

Compute the sample covariance matrix of the centered data:

$$C = \frac{1}{n-1} X_c^T X_c$$

where $n$ is the number of samples. We use $n-1$ (Bessel's correction) for an unbiased estimate.

In [ ]:
try:

    def compute_covariance(X_centered):
        """Compute the sample covariance matrix of centered data.

        Parameters
        ----------
        X_centered : np.ndarray of shape (n_samples, n_features)

        Returns
        -------
        cov : np.ndarray of shape (n_features, n_features)
        """
        # YOUR CODE HERE
        raise NotImplementedError("Implement compute_covariance")

    # Quick test
    X_test = np.array([[1.0, 2.0], [3.0, 4.0], [5.0, 6.0]])
    X_c, _ = center_data(X_test)
    cov = compute_covariance(X_c)
    assert cov.shape == (2, 2), f"Expected shape (2, 2), got {cov.shape}"
    assert np.allclose(cov, cov.T), "Covariance matrix should be symmetric"
    assert np.allclose(cov, np.cov(X_test.T)), "Should match numpy's np.cov"
    print("compute_covariance tests passed! ✓")
except Exception as e:
    print(f"⚠️ Skipped (not yet implemented or error): {e}")

### Exercise 2.3: Eigendecomposition

Compute the eigenvalues and eigenvectors of the covariance matrix using `np.linalg.eigh` (which is optimized for symmetric matrices), then sort them in **descending order** of eigenvalue.

The eigenvectors are the **principal components** (directions of maximum variance), and the eigenvalues represent the **variance explained** by each component.

In [ ]:
try:

    def eigendecompose(cov_matrix):
        """Eigendecompose a symmetric covariance matrix, sorted by descending eigenvalue.

        Parameters
        ----------
        cov_matrix : np.ndarray of shape (n_features, n_features)

        Returns
        -------
        eigenvalues : np.ndarray of shape (n_features,)
            Sorted in descending order.
        eigenvectors : np.ndarray of shape (n_features, n_features)
            Column i is the eigenvector corresponding to eigenvalues[i].
        """
        # YOUR CODE HERE
        raise NotImplementedError("Implement eigendecompose")

    # Quick test
    cov_test = np.array([[2.0, 1.0], [1.0, 3.0]])
    evals, evecs = eigendecompose(cov_test)
    assert len(evals) == 2, f"Expected 2 eigenvalues, got {len(evals)}"
    assert evals[0] >= evals[1], f"Eigenvalues should be sorted descending: {evals}"
    assert all(v >= -1e-10 for v in evals), f"Eigenvalues should be non-negative: {evals}"
    # Check eigenvector orthogonality
    assert np.allclose(np.abs(np.dot(evecs[:, 0], evecs[:, 1])), 0, atol=1e-10), "Eigenvectors should be orthogonal"
    print("eigendecompose tests passed! ✓")
except Exception as e:
    print(f"⚠️ Skipped (not yet implemented or error): {e}")

### Exercise 2.4: SVD Decomposition

An alternative to eigendecomposition is the **Singular Value Decomposition** of the centered data matrix:

$$X_c = U \Sigma V^T$$

The columns of $V$ (rows of $V^T$) are the principal components, and the eigenvalues of the covariance matrix can be recovered as:

$$\lambda_i = \frac{\sigma_i^2}{n - 1}$$

**Why SVD?** SVD operates directly on $X_c$ and avoids forming $X_c^T X_c$, which squares the condition number and can cause numerical instability.

In [ ]:
try:

    def svd_decompose(X_centered):
        """Compute the compact SVD of the centered data matrix.

        Parameters
        ----------
        X_centered : np.ndarray of shape (n_samples, n_features)

        Returns
        -------
        U : np.ndarray of shape (n_samples, min(n_samples, n_features))
        S : np.ndarray of shape (min(n_samples, n_features),)
            Singular values (sorted descending by numpy convention).
        Vt : np.ndarray of shape (min(n_samples, n_features), n_features)
            Rows are the right singular vectors (principal directions).
        """
        # YOUR CODE HERE
        raise NotImplementedError("Implement svd_decompose")

    # Quick test
    X_test = np.array([[1.0, 2.0], [3.0, 4.0], [5.0, 6.0]])
    X_c, _ = center_data(X_test)
    U, S, Vt = svd_decompose(X_c)
    assert U.shape == (3, 2), f"U shape should be (3, 2), got {U.shape}"
    assert S.shape == (2,), f"S shape should be (2,), got {S.shape}"
    assert Vt.shape == (2, 2), f"Vt shape should be (2, 2), got {Vt.shape}"
    # Verify reconstruction
    X_reconstructed = U @ np.diag(S) @ Vt
    assert np.allclose(X_c, X_reconstructed), "U @ diag(S) @ Vt should reconstruct X_centered"
    print("svd_decompose tests passed! ✓")
except Exception as e:
    print(f"⚠️ Skipped (not yet implemented or error): {e}")

### Exercise 2.5: Project Data

Project the centered data onto the top-$k$ principal components:

$$Z = X_c \cdot V_k^T$$

where $V_k$ is a matrix whose **rows** are the top-$k$ components (shape `(k, d)`).

In [ ]:
try:

    def project_data(X_centered, components, n_components):
        """Project centered data onto the top-k principal components.

        Parameters
        ----------
        X_centered : np.ndarray of shape (n_samples, n_features)
        components : np.ndarray of shape (n_all_components, n_features)
            Rows are principal components (sorted by importance).
        n_components : int
            Number of components to project onto.

        Returns
        -------
        X_projected : np.ndarray of shape (n_samples, n_components)
        """
        # YOUR CODE HERE
        raise NotImplementedError("Implement project_data")

    # Quick test
    X_test = np.array([[1.0, 2.0, 3.0], [4.0, 5.0, 6.0]])
    components_test = np.eye(3)  # Identity = no rotation
    X_proj = project_data(X_test, components_test, 2)
    assert X_proj.shape == (2, 2), f"Expected shape (2, 2), got {X_proj.shape}"
    assert np.allclose(X_proj, X_test[:, :2]), "Projection with identity should select first 2 cols"
    print("project_data tests passed! ✓")
except Exception as e:
    print(f"⚠️ Skipped (not yet implemented or error): {e}")

### Exercise 2.6: Reconstruct Data

Reconstruct the original data from the projected (lower-dimensional) representation:

$$\hat{X} = Z \cdot V_k + \bar{X}$$

where $V_k$ is the top-$k$ components matrix (shape `(k, d)`) and $\bar{X}$ is the original mean.

In [ ]:
try:

    def reconstruct_data(X_projected, components, mean):
        """Reconstruct data from its projected representation.

        Parameters
        ----------
        X_projected : np.ndarray of shape (n_samples, n_components)
        components : np.ndarray of shape (n_components, n_features)
            The top-k components used for projection.
        mean : np.ndarray of shape (n_features,)
            The original data mean.

        Returns
        -------
        X_reconstructed : np.ndarray of shape (n_samples, n_features)
        """
        # YOUR CODE HERE
        raise NotImplementedError("Implement reconstruct_data")

    # Quick test
    X_test = np.array([[1.0, 2.0, 3.0], [4.0, 5.0, 6.0]])
    X_c, mu = center_data(X_test)
    comp = np.eye(3)[:2]  # First 2 components
    X_proj = project_data(X_c, np.eye(3), 2)
    X_recon = reconstruct_data(X_proj, comp, mu)
    assert X_recon.shape == X_test.shape, f"Expected shape {X_test.shape}, got {X_recon.shape}"
    # With 2 of 3 components, reconstruction is approximate (3rd dimension lost)
    print(f"Reconstruction MSE: {np.mean((X_test - X_recon) ** 2):.4f}")
    print("reconstruct_data tests passed! ✓")
except Exception as e:
    print(f"⚠️ Skipped (not yet implemented or error): {e}")

## Section 3: PCA Classes

Now let's combine the building blocks into two full PCA classes:
1. **PCAEigen** — uses eigendecomposition of the covariance matrix
2. **PCASVD** — uses SVD of the centered data matrix

Both should produce the same results (up to sign flips in components), but SVD is preferred in practice for numerical stability.

### Exercise 3.1: PCAEigen

Implement PCA using eigendecomposition. The class should store:
- `self.components_` — shape `(n_components, n_features)`, the top principal components as rows
- `self.explained_variance_ratio_` — shape `(n_components,)`, fraction of total variance explained
- `self.mean_` — shape `(n_features,)`, the column-wise mean used for centering

Use the building blocks from Section 2: `center_data`, `compute_covariance`, `eigendecompose`, `project_data`, `reconstruct_data`.

In [ ]:
class PCAEigen:
    """PCA via eigendecomposition of the covariance matrix.

    Parameters
    ----------
    n_components : int
        Number of principal components to keep.
    """

    def __init__(self, n_components=2):
        self.n_components = n_components
        self.components_ = None
        self.explained_variance_ratio_ = None
        self.mean_ = None

    def fit(self, X):
        """Fit PCA on the data.

        Parameters
        ----------
        X : np.ndarray of shape (n_samples, n_features)

        Returns
        -------
        self
        """
        # TODO: Center the data, compute covariance, eigendecompose,
        # store components_, explained_variance_ratio_, and mean_
        raise NotImplementedError("Implement PCAEigen.fit")

    def transform(self, X):
        """Project data onto the principal components.

        Parameters
        ----------
        X : np.ndarray of shape (n_samples, n_features)

        Returns
        -------
        X_projected : np.ndarray of shape (n_samples, n_components)
        """
        # TODO: Center X using self.mean_, then project using self.components_
        raise NotImplementedError("Implement PCAEigen.transform")

    def inverse_transform(self, X_projected):
        """Reconstruct data from projected representation.

        Parameters
        ----------
        X_projected : np.ndarray of shape (n_samples, n_components)

        Returns
        -------
        X_reconstructed : np.ndarray of shape (n_samples, n_features)
        """
        # TODO: Reconstruct using self.components_ and self.mean_
        raise NotImplementedError("Implement PCAEigen.inverse_transform")

    def fit_transform(self, X):
        """Fit and transform in one step."""
        return self.fit(X).transform(X)

In [ ]:
try:
    run_single_test(PCAEigen, check_pca_eigen)
except Exception as e:
    print(f"⚠️ Skipped (not yet implemented or error): {e}")

In [ ]:
try:
    X_blobs, y_blobs = generate_high_dim_blobs()
    pca_eigen = PCAEigen(n_components=10)
    X_proj = pca_eigen.fit_transform(X_blobs)
    X_2d = X_proj[:, :2]
    plot_2d_projection(X_2d, y_blobs, title="PCAEigen: 2D Projection of 20D Data")
    plot_explained_variance(pca_eigen.explained_variance_ratio_, title="PCAEigen: Explained Variance")
except Exception as e:
    print(f"⚠️ Skipped (not yet implemented or error): {e}")

### Exercise 3.2: PCASVD

Implement PCA using SVD. The key differences from PCAEigen:

- Instead of computing the covariance matrix and its eigenvectors, use `svd_decompose` on the centered data
- The principal components are the rows of $V^T$: `components_ = Vt[:n_components]`
- The eigenvalues can be recovered from singular values: $\lambda_i = \frac{\sigma_i^2}{n - 1}$

The class should have the same API as PCAEigen.

In [ ]:
class PCASVD:
    """PCA via Singular Value Decomposition of the centered data matrix.

    Parameters
    ----------
    n_components : int
        Number of principal components to keep.
    """

    def __init__(self, n_components=2):
        self.n_components = n_components
        self.components_ = None
        self.explained_variance_ratio_ = None
        self.mean_ = None

    def fit(self, X):
        """Fit PCA on the data using SVD.

        Parameters
        ----------
        X : np.ndarray of shape (n_samples, n_features)

        Returns
        -------
        self
        """
        # TODO: Center data, compute SVD, extract components and eigenvalues
        raise NotImplementedError("Implement PCASVD.fit")

    def transform(self, X):
        """Project data onto the principal components.

        Parameters
        ----------
        X : np.ndarray of shape (n_samples, n_features)

        Returns
        -------
        X_projected : np.ndarray of shape (n_samples, n_components)
        """
        # TODO: Center X using self.mean_, then project using self.components_
        raise NotImplementedError("Implement PCASVD.transform")

    def inverse_transform(self, X_projected):
        """Reconstruct data from projected representation.

        Parameters
        ----------
        X_projected : np.ndarray of shape (n_samples, n_components)

        Returns
        -------
        X_reconstructed : np.ndarray of shape (n_samples, n_features)
        """
        # TODO: Reconstruct using self.components_ and self.mean_
        raise NotImplementedError("Implement PCASVD.inverse_transform")

    def fit_transform(self, X):
        """Fit and transform in one step."""
        return self.fit(X).transform(X)

In [ ]:
try:
    run_single_test(PCASVD, check_pca_svd)
except Exception as e:
    print(f"⚠️ Skipped (not yet implemented or error): {e}")

### Eigendecomposition vs SVD

Both methods should yield the **same principal components** (up to sign flips) and **identical eigenvalues**.

However, SVD is preferred in practice because:
1. It avoids forming $X^T X$, which **squares the condition number**
2. It is more numerically stable for ill-conditioned data
3. It can handle cases where $n < d$ more efficiently

Let's verify the equivalence:

In [ ]:
try:
    X_blobs, y_blobs = generate_high_dim_blobs()

    pca_e = PCAEigen(n_components=10).fit(X_blobs)
    pca_s = PCASVD(n_components=10).fit(X_blobs)

    # Recover full eigenvalues for comparison
    X_c, _ = center_data(X_blobs)
    cov = compute_covariance(X_c)
    evals_eigen, _ = eigendecompose(cov)

    _, S, _ = svd_decompose(X_c)
    evals_svd = S**2 / (len(X_blobs) - 1)

    plot_pca_eigen_vs_svd(evals_eigen, evals_svd)

    # Component alignment
    print("\nComponent alignment (|dot product| — should be ~1.0):")
    for i in range(min(5, pca_e.n_components)):
        dot = abs(np.dot(pca_e.components_[i], pca_s.components_[i]))
        print(f"  Component {i + 1}: {dot:.6f}")
except Exception as e:
    print(f"⚠️ Skipped (not yet implemented or error): {e}")

## Section 4: Autoencoder Building Blocks

An **autoencoder** is a neural network trained to reconstruct its own input through a bottleneck:

$$\text{Input} \xrightarrow{\text{Encoder}} \text{Latent} \xrightarrow{\text{Decoder}} \text{Reconstruction}$$

The bottleneck forces the network to learn a compressed representation. When the encoder and decoder are **linear** (no activation functions), the autoencoder learns the same subspace as PCA!

With **non-linear** activations (e.g., ReLU), the autoencoder can learn complex manifold structures that PCA cannot capture.

### Exercise 4.1: Reconstruction Loss

The standard autoencoder loss is **Mean Squared Error (MSE)**:

$$\mathcal{L} = \frac{1}{n} \sum_{i=1}^{n} \| x_i - \hat{x}_i \|^2$$

In [ ]:
try:

    def reconstruction_loss(X_original, X_reconstructed):
        """Compute the Mean Squared Error between original and reconstructed data.

        Parameters
        ----------
        X_original : torch.Tensor of shape (n_samples, n_features)
        X_reconstructed : torch.Tensor of shape (n_samples, n_features)

        Returns
        -------
        loss : torch.Tensor (scalar)
        """
        # YOUR CODE HERE
        raise NotImplementedError("Implement reconstruction_loss")

    # Quick test
    X_orig = torch.tensor([[1.0, 2.0], [3.0, 4.0]])
    X_recon = torch.tensor([[1.1, 1.9], [3.2, 3.8]])
    loss = reconstruction_loss(X_orig, X_recon)
    assert loss.dim() == 0, f"Loss should be a scalar, got shape {loss.shape}"
    expected = torch.mean((X_orig - X_recon) ** 2)
    assert torch.allclose(loss, expected), f"Expected {expected.item():.4f}, got {loss.item():.4f}"
    print(f"reconstruction_loss = {loss.item():.4f}")
    print("reconstruction_loss tests passed! ✓")
except Exception as e:
    print(f"⚠️ Skipped (not yet implemented or error): {e}")

### Exercise 4.2: Training Loop

Implement a training loop for autoencoders using the Adam optimizer and MSE loss.

The function should:
1. Convert numpy data to a torch tensor
2. For each epoch: forward pass, compute MSE loss, backward pass, optimizer step
3. Return the list of losses

In [ ]:
try:

    def train_autoencoder(model, X_np, n_epochs=300, lr=1e-3):
        """Train an autoencoder model.

        Parameters
        ----------
        model : nn.Module
            Autoencoder with forward(x) -> x_reconstructed.
        X_np : np.ndarray of shape (n_samples, n_features)
            Training data.
        n_epochs : int
            Number of training epochs.
        lr : float
            Learning rate for Adam optimizer.

        Returns
        -------
        losses : list of float
            Training loss at each epoch.
        """
        # YOUR CODE HERE
        raise NotImplementedError("Implement train_autoencoder")

    # Quick test with a trivial identity-like model
    class _TinyAE(nn.Module):
        def __init__(self):
            super().__init__()
            self.enc = nn.Linear(2, 2)
            self.dec = nn.Linear(2, 2)

        def forward(self, x):
            return self.dec(self.enc(x))

    torch.manual_seed(42)
    _test_model = _TinyAE()
    _test_data = np.random.randn(50, 2).astype(np.float32)
    _test_losses = train_autoencoder(_test_model, _test_data, n_epochs=50, lr=1e-2)
    assert len(_test_losses) == 50, f"Expected 50 losses, got {len(_test_losses)}"
    assert _test_losses[-1] < _test_losses[0], "Loss should decrease during training"
    print(f"Loss: {_test_losses[0]:.4f} -> {_test_losses[-1]:.4f}")
    print("train_autoencoder tests passed! ✓")
except Exception as e:
    print(f"⚠️ Skipped (not yet implemented or error): {e}")

## Section 5: Linear Autoencoder

A **linear autoencoder** uses no activation functions — just linear transformations:

$$z = W_e x + b_e \quad \text{(encoder)}$$
$$\hat{x} = W_d z + b_d \quad \text{(decoder)}$$

**Key insight:** When trained with MSE loss, the optimal weight matrix of a linear autoencoder spans the **same subspace** as the top PCA components! The individual weight vectors may differ (they are not necessarily the eigenvectors), but the subspace they define is identical.

This means a linear autoencoder is *at most* as powerful as PCA for dimensionality reduction.

### Exercise 5.1: LinearAutoencoder

Implement a linear autoencoder with:
- **Encoder**: `nn.Linear(input_dim, latent_dim)` — no activation
- **Decoder**: `nn.Linear(latent_dim, input_dim)` — no activation

The class must have `encode(x)`, `decode(z)`, and `forward(x)` methods.

In [ ]:
class LinearAutoencoder(nn.Module):
    """Autoencoder with linear (no activation) encoder and decoder.

    Parameters
    ----------
    input_dim : int
        Dimensionality of input features.
    latent_dim : int
        Dimensionality of the bottleneck (latent space).
    """

    def __init__(self, input_dim, latent_dim):
        super().__init__()
        # TODO: Create self.encoder and self.decoder as nn.Linear layers
        raise NotImplementedError("Implement LinearAutoencoder.__init__")

    def encode(self, x):
        """Encode input to latent representation.

        Parameters
        ----------
        x : torch.Tensor of shape (batch_size, input_dim)

        Returns
        -------
        z : torch.Tensor of shape (batch_size, latent_dim)
        """
        # YOUR CODE HERE
        raise NotImplementedError("Implement LinearAutoencoder.encode")

    def decode(self, z):
        """Decode latent representation to reconstruction.

        Parameters
        ----------
        z : torch.Tensor of shape (batch_size, latent_dim)

        Returns
        -------
        x_recon : torch.Tensor of shape (batch_size, input_dim)
        """
        # YOUR CODE HERE
        raise NotImplementedError("Implement LinearAutoencoder.decode")

    def forward(self, x):
        """Full forward pass: encode then decode.

        Parameters
        ----------
        x : torch.Tensor of shape (batch_size, input_dim)

        Returns
        -------
        x_recon : torch.Tensor of shape (batch_size, input_dim)
        """
        # YOUR CODE HERE
        raise NotImplementedError("Implement LinearAutoencoder.forward")

In [ ]:
try:
    run_single_test(LinearAutoencoder, check_linear_autoencoder)
except Exception as e:
    print(f"⚠️ Skipped (not yet implemented or error): {e}")

In [ ]:
try:
    X_blobs, y_blobs = generate_high_dim_blobs()
    X_mean = X_blobs.mean(axis=0)
    X_std = X_blobs.std(axis=0) + 1e-8
    X_norm = (X_blobs - X_mean) / X_std

    torch.manual_seed(42)
    lae = LinearAutoencoder(input_dim=20, latent_dim=3)
    lae_losses = train_autoencoder(lae, X_norm, n_epochs=2000, lr=1e-2)
    plot_autoencoder_training(lae_losses, title="Linear Autoencoder Training")
except Exception as e:
    print(f"⚠️ Skipped (not yet implemented or error): {e}")

### PCA Equivalence Demonstration

Let's verify that the linear autoencoder learns the same subspace as PCA.

We project data through both the AE encoder and PCA, then find the best linear rotation between the two latent spaces using least squares. If the subspaces match, the **R²** (coefficient of determination) should be close to 1.

In [ ]:
try:
    # Project data through AE encoder
    lae.eval()
    with torch.no_grad():
        X_t = torch.tensor(X_norm, dtype=torch.float32)
        z_ae = lae.encode(X_t).numpy()  # (n, 3)

    # Project data through PCA
    pca = PCAEigen(n_components=3).fit(X_norm)
    X_c = X_norm - X_norm.mean(axis=0)
    z_pca = X_c @ pca.components_.T  # (n, 3)

    # Find best rotation from AE latent space to PCA latent space
    R, _, _, _ = np.linalg.lstsq(z_ae, z_pca, rcond=None)
    z_ae_aligned = z_ae @ R
    ss_res = np.sum((z_pca - z_ae_aligned) ** 2)
    ss_tot = np.sum((z_pca - z_pca.mean(axis=0)) ** 2)
    r_squared = 1 - ss_res / ss_tot

    print(f"Subspace R² = {r_squared:.4f}")
    if r_squared > 0.95:
        print("\nThe linear autoencoder learned the same subspace as PCA!")
    else:
        print("\nThe subspaces are not fully aligned yet. Try training longer.")
except Exception as e:
    print(f"⚠️ Skipped (not yet implemented or error): {e}")

## Section 6: Non-linear Autoencoder

By adding **non-linear activations** (ReLU) between layers, the autoencoder can learn **complex, curved manifolds** that linear methods like PCA cannot capture.

Architecture:
```
Encoder: input_dim -> hidden_dim (ReLU) -> latent_dim
Decoder: latent_dim -> hidden_dim (ReLU) -> input_dim
```

We'll demonstrate this on the **Swiss roll** — a 2D manifold embedded in 3D that PCA cannot unfold.

In [ ]:
try:
    X_swiss, color_swiss = generate_swiss_roll()
    plot_swiss_roll_3d(X_swiss, color_swiss, title="Swiss Roll (3D)")
    print(f"Swiss roll shape: {X_swiss.shape}")
    print(f"The data lives on a 2D manifold embedded in 3D space.")
except Exception as e:
    print(f"⚠️ Skipped (not yet implemented or error): {e}")

### Exercise 6.1: NonlinearAutoencoder

Implement a non-linear autoencoder with ReLU activations:

**Encoder:** `input_dim -> hidden_dim` (ReLU) `-> latent_dim`

**Decoder:** `latent_dim -> hidden_dim` (ReLU) `-> input_dim`

The class must have `encode(x)`, `decode(z)`, and `forward(x)` methods.

In [ ]:
class NonlinearAutoencoder(nn.Module):
    """Autoencoder with ReLU activations for non-linear manifold learning.

    Parameters
    ----------
    input_dim : int
        Dimensionality of input features.
    hidden_dim : int
        Dimensionality of hidden layers.
    latent_dim : int
        Dimensionality of the bottleneck (latent space).
    """

    def __init__(self, input_dim, hidden_dim, latent_dim):
        super().__init__()
        # TODO: Create self.encoder and self.decoder using nn.Sequential
        raise NotImplementedError("Implement NonlinearAutoencoder.__init__")

    def encode(self, x):
        """Encode input to latent representation.

        Parameters
        ----------
        x : torch.Tensor of shape (batch_size, input_dim)

        Returns
        -------
        z : torch.Tensor of shape (batch_size, latent_dim)
        """
        # YOUR CODE HERE
        raise NotImplementedError("Implement NonlinearAutoencoder.encode")

    def decode(self, z):
        """Decode latent representation to reconstruction.

        Parameters
        ----------
        z : torch.Tensor of shape (batch_size, latent_dim)

        Returns
        -------
        x_recon : torch.Tensor of shape (batch_size, input_dim)
        """
        # YOUR CODE HERE
        raise NotImplementedError("Implement NonlinearAutoencoder.decode")

    def forward(self, x):
        """Full forward pass: encode then decode.

        Parameters
        ----------
        x : torch.Tensor of shape (batch_size, input_dim)

        Returns
        -------
        x_recon : torch.Tensor of shape (batch_size, input_dim)
        """
        # YOUR CODE HERE
        raise NotImplementedError("Implement NonlinearAutoencoder.forward")

In [ ]:
try:
    run_single_test(NonlinearAutoencoder, check_nonlinear_autoencoder)
except Exception as e:
    print(f"⚠️ Skipped (not yet implemented or error): {e}")

In [ ]:
try:
    # Standardize Swiss roll
    X_swiss_mean = X_swiss.mean(axis=0)
    X_swiss_std = X_swiss.std(axis=0) + 1e-8
    X_swiss_norm = (X_swiss - X_swiss_mean) / X_swiss_std

    torch.manual_seed(42)
    nae = NonlinearAutoencoder(input_dim=3, hidden_dim=64, latent_dim=2)
    nae_losses = train_autoencoder(nae, X_swiss_norm, n_epochs=500, lr=1e-3)
    plot_autoencoder_training(nae_losses, title="Non-linear Autoencoder on Swiss Roll")
except Exception as e:
    print(f"⚠️ Skipped (not yet implemented or error): {e}")

In [ ]:
try:
    nae.eval()
    with torch.no_grad():
        X_t = torch.tensor(X_swiss_norm, dtype=torch.float32)
        z_swiss = nae.encode(X_t).numpy()

    plot_latent_space(z_swiss, color_swiss, title="Non-linear AE: Swiss Roll Latent Space")
except Exception as e:
    print(f"⚠️ Skipped (not yet implemented or error): {e}")

### PCA vs Non-linear Autoencoder on Swiss Roll

PCA projects the Swiss roll onto a flat plane, which crushes the manifold structure. The non-linear autoencoder can "unfold" the roll by learning the underlying 2D manifold.

In [ ]:
try:
    # PCA 2D projection of Swiss roll
    pca_swiss = PCASVD(n_components=2).fit(X_swiss_norm)
    X_pca_2d = pca_swiss.transform(X_swiss_norm)

    # Compare
    plot_manifold_comparison(X_pca_2d, z_swiss, color_swiss, titles=("PCA 2D (linear)", "Non-linear AE 2D"))

    # Reconstruction errors
    X_pca_recon = pca_swiss.inverse_transform(X_pca_2d)
    pca_mse = np.mean((X_swiss_norm - X_pca_recon) ** 2)

    nae.eval()
    with torch.no_grad():
        X_t = torch.tensor(X_swiss_norm, dtype=torch.float32)
        nae_recon = nae(X_t).numpy()
    nae_mse = np.mean((X_swiss_norm - nae_recon) ** 2)

    print(f"PCA reconstruction MSE:      {pca_mse:.4f}")
    print(f"Non-linear AE reconstruction MSE: {nae_mse:.4f}")
    if nae_mse < pca_mse:
        print("Non-linear AE achieves better reconstruction — it captures the curved manifold!")
except Exception as e:
    print(f"⚠️ Skipped (not yet implemented or error): {e}")

## Section 7: Summary & Key Takeaways

| Method | Type | Captures | Strengths |
|--------|------|----------|----------|
| PCA (Eigen) | Linear | Directions of max variance | Closed-form, interpretable |
| PCA (SVD) | Linear | Same as Eigen | Numerically stable |
| Linear AE | Linear | Same subspace as PCA | Shows NN-PCA connection |
| Non-linear AE | Non-linear | Curved manifolds | Handles complex structure |

**Key insights:**
1. PCA via eigendecomposition and SVD give the **same results** — SVD is preferred for stability
2. A **linear autoencoder** learns the **PCA subspace** — no benefit over PCA
3. **Non-linear autoencoders** can capture manifold structure that PCA misses (e.g., Swiss roll)
4. The reconstruction error quantifies how much information is preserved in the low-dimensional representation

## Section 8: Run All Tests

In [ ]:
try:
    run_tests(
        PCAEigenClass=PCAEigen,
        PCASVDClass=PCASVD,
        LinearAutoencoderClass=LinearAutoencoder,
        NonlinearAutoencoderClass=NonlinearAutoencoder,
    )
except Exception as e:
    print(f"⚠️ Skipped (not yet implemented or error): {e}")